Las features y el nombre del archivo donde se guardan los resultados están hardcodeadas en el archivo model.py y main.py (perdón por eso)

In [1]:
# Cambiar el directorio de trabajo a la carpeta de SASRec
%cd SASRec.pytorch/python

# Ejecutar el entrenamiento usando el nuevo txt generado
!python main.py --dataset=interaction_resumen --train_dir=mis_resultados --maxlen=200 --hidden_units=64 --dropout_rate=0.2 --device=cuda --num_epochs=100

# Volver al directorio original
%cd ../..

/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/Sistemas_Recomendadores/repo_proyecto/recsys-pf/notebooks/SASRec.pytorch/python
average sequence length: 2.79

[!] Éxito: Pesos de items cargados desde 'embeddings_efmrec_gated.pt'
loss in epoch 1 iteration 0: 1.3901238441467285
loss in epoch 1 iteration 1: 1.3909448385238647
loss in epoch 1 iteration 2: 1.3863232135772705
loss in epoch 1 iteration 3: 1.3862390518188477
loss in epoch 1 iteration 4: 1.3806514739990234
loss in epoch 1 iteration 5: 1.3848154544830322
loss in epoch 1 iteration 6: 1.3789674043655396
loss in epoch 1 iteration 7: 1.3705973625183105
loss in epoch 1 iteration 8: 1.3819748163223267
loss in epoch 1 iteration 9: 1.3784431219100952
loss in epoch 1 iteration 10: 1.3715977668762207
loss in epoch 1 iteration 11: 1.3739997148513794
loss in epoch 1 iteration 12: 1.3755271434783936
loss in epoch 1 iteration 13: 1.3719217777252197
loss in epoch 1 iteration 14: 1.3688952922821045
loss in epoch 1 iteration 15: 1.369

# Evaluar resultados

In [2]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import glob
import gdown
import zipfile
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
import scipy.sparse as sp
import torch.optim as optim
import random
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display
import copy
import gc
import json

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/Sistemas_Recomendadores/recomendadores/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1782661138.913794    9176 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782661138.958972    9176 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/S

In [3]:
# cargar datos
interaction_df = pd.read_csv('recsys-pf/interaction.csv')
item_info_df = pd.read_csv('recsys-pf/item_info.csv')
item_info_df = item_info_df.dropna()

n_items_to_sample = 15000
sampled_item_info_df = item_info_df.sample(n=n_items_to_sample, random_state=5)

valid_item_ids = sampled_item_info_df['item_id'].unique()

sampled_interaction_df = interaction_df[interaction_df['item_id'].isin(valid_item_ids)]

image_folder = 'recsys-pf/resized_images'
valid_image_paths = [os.path.join(image_folder, f"{item_id}.jpg") for item_id in valid_item_ids]

print(f"Sampled items: {len(sampled_item_info_df)}")
print(f"Sampled interactions: {len(sampled_interaction_df)}")
print(f"Number of valid image paths: {len(valid_image_paths)}")

sampled_interaction_df.loc[:, 'user_id'] = sampled_interaction_df['user_id'].astype('category').cat.codes
sampled_interaction_df.loc[:, 'item_id'] = sampled_interaction_df['item_id'].astype('category').cat.codes

num_users = sampled_interaction_df['user_id'].nunique()
num_items = sampled_interaction_df['item_id'].nunique()

sampled_interaction_df = sampled_interaction_df.sort_values('timestamp')
total_rows = len(sampled_interaction_df)
train_split_idx = int(total_rows * 0.80)
val_split_idx = int(total_rows * 0.90)
train_df = sampled_interaction_df.iloc[:train_split_idx].copy()
val_df = sampled_interaction_df.iloc[train_split_idx:val_split_idx].copy()
test_df = sampled_interaction_df.iloc[val_split_idx:].copy()

train_df['user_id'] = train_df['user_id'].astype(int)
train_df['item_id'] = train_df['item_id'].astype(int)
val_df['user_id'] = val_df['user_id'].astype(int)
val_df['item_id'] = val_df['item_id'].astype(int)
test_df['user_id'] = test_df['user_id'].astype(int)
test_df['item_id'] = test_df['item_id'].astype(int)


Sampled items: 15000
Sampled interactions: 188108
Number of valid image paths: 15000


In [ ]:
# Modifcar interacciones para SASRec
# Debe ser un archivo txt, con solo dos números (user_id item_id) separados por espacio ordenados cronologicamente por usuario (que asumo ya se hizo)

df_sasrec = train_df[['user_id', 'item_id']].copy()

usuarios_unicos = df_sasrec['user_id'].unique()
user2id_denso = {user: i + 1 for i, user in enumerate(usuarios_unicos)}
id2user_original = {i + 1: user for i, user in enumerate(usuarios_unicos)}

df_sasrec['user_id'] = df_sasrec['user_id'].map(user2id_denso)
df_sasrec['item_id'] = df_sasrec['item_id'] + 1

ruta_salida = 'SASRec.pytorch/python/data'
os.makedirs(ruta_salida, exist_ok=True)
ruta_txt = os.path.join(ruta_salida, 'interaction_resumen.txt')

df_sasrec.sort_values(by=['user_id', 'item_id']).to_csv(ruta_txt, sep=' ', index=False, header=False)
print(f"Se guardaron las interacciones para SASRec en: {ruta_txt}")

In [ ]:
id2user_original_json = {int(k): int(v) for k, v in id2user_original.items()}

ruta_json = os.path.join(ruta_salida, 'mapa_usuarios.json')
with open(ruta_json, 'w') as f:
    json.dump(id2user_original_json, f)

print(f"Diccionario de usuarios para evaluación: {ruta_json}")

In [4]:
# De práctico_métricas.ipynb
def precision_at_k(r, k):
    assert 1 <= k <= r.size
    return (np.asarray(r)[:k] != 0).mean()

def average_precision_at_k(r, k):
    r = np.asarray(r)
    score = 0.
    for i in range(min(k, r.size)):
        score += precision_at_k(r, i + 1)
    return score / k

def dcg_at_k(r, k):
    r = np.asarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.

def idcg_at_k(k):
    return dcg_at_k(np.ones(k), k)

def ndcg_at_k(r, k, max_relevant):
    idcg = idcg_at_k(min(k, max_relevant))
    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg

def recall_at_k(relevant_items, recommended_items, k):
    relevant_items = set(relevant_items)
    recommended_items = set(recommended_items[:k])
    intersection = relevant_items.intersection(recommended_items)
    recall = len(intersection) / len(relevant_items) if len(relevant_items) > 0 else 0
    return recall

def evaluate_model(y_true, y_pred, N):
    map_scores = 0.
    ndcg_scores = 0.
    recall_scores = 0.
    
    for true_items, pred_items in zip(y_true, y_pred):
        
        true_set = set(true_items)
        
        # el relevance vector
        r = [1 if item in true_set else 0 for item in pred_items]
        
        # MAP
        user_map = average_precision_at_k(r, N)
        map_scores += user_map
        # NDGC@N
        max_relevant = len(true_items)
        user_ndcg = ndcg_at_k(r, N, max_relevant)
        ndcg_scores += user_ndcg
        # Recall@N
        user_recall = recall_at_k(true_items, pred_items, N)
        recall_scores += user_recall
        
    cantidad = len(y_true)
    final_map = map_scores / cantidad
    final_ndcg = ndcg_scores / cantidad
    final_recall = recall_scores / cantidad
    
    print(f"Métricas de evaluación ranking (Top-{N}):")
    print(f"MAP@{N}:    {final_map:.5f}")
    print(f"nDCG@{N}:   {final_ndcg:.5f}")
    print(f"Recall@{N}: {final_recall:.5f}")
    
    return final_map, final_ndcg, final_recall

In [5]:
def evaluar_sasrec(ruta_csv_sasrec, test_df, N=10):
    test_dict = test_df.groupby('user_id')['item_id'].apply(list).to_dict()

    y_true = []
    y_pred = []

    sasrec = pd.read_csv(ruta_csv_sasrec)

    with open('SASRec.pytorch/python/data/mapa_usuarios.json', 'r') as f:
        diccionario_cargado = json.load(f)
    id2user_original = {int(k): int(v) for k, v in diccionario_cargado.items()}

    # Revertir el mapeo de usuarios
    sasrec['user_id'] = sasrec['user_id'].map(id2user_original)

    # Revertir el ajuste de ítems (restamos 1 a todas las columnas de rec_)
    for i in range(1, N + 1):
        sasrec[f'rec_{i}'] = sasrec[f'rec_{i}'] - 1

    sasrec_set = sasrec.set_index('user_id')

    for user_id_num, true_items in test_dict.items():
        if user_id_num in sasrec_set.index:
            row = sasrec_set.loc[user_id_num]
            lista_predicha = row[[f'rec_{i}' for i in range(1, N+1)]].tolist()
            
            if len(lista_predicha) > 0:
                y_true.append(true_items)
                y_pred.append(lista_predicha)

    map_score, ndcg_score, recall_score = evaluate_model(y_true, y_pred, N)
    
    return map_score, ndcg_score, recall_score

In [8]:
comparacion = pd.DataFrame(columns=["modelo", "map", "ndcg", "recall"])
map_clip, ndcg_clip, recall_clip = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_100.csv",
    test_df=test_df,
    N=10
)
comparacion.loc[0] = ["clip@10", map_clip, ndcg_clip, recall_clip]

map_clip_gated, ndcg_clip_gated, recall_clip_gated = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_gated_100.csv",
    test_df=test_df,
    N=10
)
comparacion.loc[1] = ["clip-gated@10", map_clip_gated, ndcg_clip_gated, recall_clip_gated]

map_siglip, ndcg_siglip, recall_siglip = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_siglip_100.csv",
    test_df=test_df,
    N=10
)
comparacion.loc[2] = ["siglip@10", map_siglip, ndcg_siglip, recall_siglip]

map_siglip_gated, ndcg_siglip_gated, recall_siglip_gated = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_siglip_gated_100.csv",
    test_df=test_df,
    N=10
)
comparacion.loc[3] = ["sglip-gated@10", map_siglip_gated, ndcg_siglip_gated, recall_siglip_gated]


Métricas de evaluación ranking (Top-10):
MAP@10:    0.00022
nDCG@10:   0.00086
Recall@10: 0.00205
Métricas de evaluación ranking (Top-10):
MAP@10:    0.00022
nDCG@10:   0.00068
Recall@10: 0.00144
Métricas de evaluación ranking (Top-10):
MAP@10:    0.00019
nDCG@10:   0.00062
Recall@10: 0.00129
Métricas de evaluación ranking (Top-10):
MAP@10:    0.00023
nDCG@10:   0.00081
Recall@10: 0.00171


In [9]:
display(comparacion)

,modelo,map,ndcg,recall
0,clip@10,0.000221,0.000857,0.002051
1,clip-gated@10,0.000222,0.000683,0.001443
2,siglip@10,0.000192,0.000615,0.001287
3,sglip-gated@10,0.000229,0.000810,0.001713


In [11]:
comparacion5 = pd.DataFrame(columns=["modelo", "map", "ndcg", "recall"])
map_clip, ndcg_clip, recall_clip = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_100.csv",
    test_df=test_df,
    N=5
)
comparacion5.loc[0] = ["clip@10", map_clip, ndcg_clip, recall_clip]

map_clip_gated, ndcg_clip_gated, recall_clip_gated = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_gated_100.csv",
    test_df=test_df,
    N=5
)
comparacion5.loc[1] = ["clip-gated@10", map_clip_gated, ndcg_clip_gated, recall_clip_gated]

map_siglip, ndcg_siglip, recall_siglip = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_siglip_100.csv",
    test_df=test_df,
    N=5
)
comparacion5.loc[2] = ["siglip@10", map_siglip, ndcg_siglip, recall_siglip]

map_siglip_gated, ndcg_siglip_gated, recall_siglip_gated = evaluar_sasrec(
    ruta_csv_sasrec="SASRec.pytorch/python/recomendaciones_sasrec_siglip_gated_100.csv",
    test_df=test_df,
    N=5
)
comparacion5.loc[3] = ["sglip-gated@10", map_siglip_gated, ndcg_siglip_gated, recall_siglip_gated]

display(comparacion5)

Métricas de evaluación ranking (Top-5):
MAP@5:    0.00016
nDCG@5:   0.00038
Recall@5: 0.00069
Métricas de evaluación ranking (Top-5):
MAP@5:    0.00022
nDCG@5:   0.00054
Recall@5: 0.00101
Métricas de evaluación ranking (Top-5):
MAP@5:    0.00017
nDCG@5:   0.00046
Recall@5: 0.00086
Métricas de evaluación ranking (Top-5):
MAP@5:    0.00021
nDCG@5:   0.00054
Recall@5: 0.00098


,modelo,map,ndcg,recall
0,clip@10,0.000160,0.000381,0.000690
1,clip-gated@10,0.000224,0.000543,0.001011
2,siglip@10,0.000166,0.000459,0.000856
3,sglip-gated@10,0.000208,0.000542,0.000978
